# 📌 MODUL 06 — Pertemuan 07: Studi Kasus Terintegrasi Retail Triwulan End-to-End
> **Tujuan Pembelajaran:** Menjalankan satu alur analisis Big Data lengkap dari awal sampai akhir persis seperti simulasi ujian responsi: Ingest ke HDFS, Extract dengan Spark, Transform (Multi-Join, Window Function Top-3, dan Spark SQL), Load ke Parquet HDFS, serta menyusun **Executive Summary** untuk pimpinan.

## 🏢 Skenario Bisnis
Sebuah jaringan retail modern memiliki cabang di 4 kota: **Magelang, Yogyakarta, Semarang, dan Solo**.  
Direksi membutuhkan laporan kinerja penjualan triwulan untuk mengevaluasi produk terlaris di tiap cabang dan kontribusi omset dari masing-masing manajer cabang.

## ⚙️ 1. Setup Environment & Inisialisasi SparkSession

In [1]:
import os
import sys
import getpass
import json
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Konfigurasi isolasi lingkungan native Arch Linux
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("Responsi-Pertemuan07-EndToEnd") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
username = os.environ.get("USER", getpass.getuser())
print(f"✅ SparkSession Aktif untuk pengguna: {username}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/01 11:18:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ SparkSession Aktif untuk pengguna: kaky


## 📦 2. Menyiapkan Tiga Sumber Data & Mengunggah ke HDFS
Kita siapkan 3 berkas:
1. `data_toko.json` (Data master toko dan nama manajer)
2. `data_produk.csv` (Katalog produk, harga, dan kategori)
3. `data_transaksi_triwulan.csv` (Catatan transaksi penjualan)

In [2]:
np.random.seed(77)

# 1. Data Toko (JSON)
data_toko = [
    {"toko_id": 1, "nama_toko": "Toko Magelang", "kota": "Magelang", "manager": "Rani Handayani"},
    {"toko_id": 2, "nama_toko": "Toko Yogyakarta", "kota": "Yogyakarta", "manager": "Joko Widodo"},
    {"toko_id": 3, "nama_toko": "Toko Semarang", "kota": "Semarang", "manager": "Sari Indah"},
    {"toko_id": 4, "nama_toko": "Toko Solo", "kota": "Solo", "manager": "Bayu Nugroho"},
]
with open("data_toko_p7.json", "w") as f:
    for item in data_toko:
        f.write(json.dumps(item) + "\n")

# 2. Data Produk (CSV)
data_produk = pd.DataFrame({
    "product_id": [101, 102, 103, 104, 105, 106],
    "nama_produk": ["Smart TV 43 Inch", "Kemeja Batik", "Kopi Susu Gula Aren", "Blender Multifungsi", "Sepatu Sneakers", "Snack Keripik Singkong"],
    "kategori": ["Elektronik", "Fashion", "Makanan", "Elektronik", "Fashion", "Makanan"],
    "harga": [3200000, 250000, 22000, 450000, 380000, 15000]
})
data_produk.to_csv("data_produk_p7.csv", index=False)

# 3. Data Transaksi (CSV) — 300 baris
n = 300
data_transaksi = pd.DataFrame({
    "order_id": [f"TX7-{i:04d}" for i in range(1, n + 1)],
    "toko_id": np.random.choice([1, 2, 3, 4], size=n),
    "product_id": np.random.choice([101, 102, 103, 104, 105, 106], size=n),
    "qty": np.random.randint(1, 8, size=n)
})
data_transaksi.to_csv("data_transaksi_p7.csv", index=False)

# Unggah semua ke HDFS
!hdfs dfs -mkdir -p /user/$USER/responsi/p7/raw
!hdfs dfs -put -f data_toko_p7.json /user/$USER/responsi/p7/raw/
!hdfs dfs -put -f data_produk_p7.csv /user/$USER/responsi/p7/raw/
!hdfs dfs -put -f data_transaksi_p7.csv /user/$USER/responsi/p7/raw/

print("✅ Ketiga sumber data berhasil diunggah ke HDFS!")
!hdfs dfs -ls -h /user/$USER/responsi/p7/raw/

mkdir: Call From alterra/127.0.1.1 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused


put: Call From alterra/127.0.1.1 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused


put: Call From alterra/127.0.1.1 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused


put: Call From alterra/127.0.1.1 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused


✅ Ketiga sumber data berhasil diunggah ke HDFS!


ls: Call From alterra/127.0.1.1 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused


## 📥 3. Tahap EXTRACT — Membaca Data dari HDFS (dengan Fallback Lokal)

In [3]:
hdfs_base = f"hdfs://localhost:9000/user/{username}/responsi/p7/raw"

try:
    df_transaksi = spark.read.csv(f"{hdfs_base}/data_transaksi_p7.csv", header=True, inferSchema=True)
    df_toko = spark.read.json(f"{hdfs_base}/data_toko_p7.json")
    df_produk = spark.read.csv(f"{hdfs_base}/data_produk_p7.csv", header=True, inferSchema=True)
    print("✅ Berhasil membaca dataset langsung dari HDFS!")
except Exception as e:
    print("⚠️ HDFS tidak aktif/terjangkau saat ini, beralih membaca langsung dari berkas lokal:")
    df_transaksi = spark.read.csv("data_transaksi_p7.csv", header=True, inferSchema=True)
    df_toko = spark.read.json("data_toko_p7.json")
    df_produk = spark.read.csv("data_produk_p7.csv", header=True, inferSchema=True)

print(f"Data Transaksi : {df_transaksi.count()} baris")
print(f"Data Toko      : {df_toko.count()} cabang")
print(f"Data Produk    : {df_produk.count()} item")

⚠️ HDFS tidak aktif/terjangkau saat ini, beralih membaca langsung dari berkas lokal:


Data Transaksi : 300 baris


Data Toko      : 4 cabang


Data Produk    : 6 item


## 🔄 4. Tahap TRANSFORM — Multi-Join & Hitung Penjualan

In [4]:
# Menggabungkan ketiga tabel
df_gabungan = df_transaksi \
    .join(df_toko, on="toko_id", how="inner") \
    .join(df_produk, on="product_id", how="inner")

# Menambahkan kolom total penjualan (qty x harga)
df_gabungan = df_gabungan.withColumn("total_penjualan", F.col("qty") * F.col("harga"))

print("Tabel Gabungan Utama (5 Baris Pertama):")
df_gabungan.select("order_id", "nama_toko", "nama_produk", "kategori", "qty", "harga", "total_penjualan").show(5)

Tabel Gabungan Utama (5 Baris Pertama):


+--------+-------------+-------------------+----------+---+------+---------------+
|order_id|    nama_toko|        nama_produk|  kategori|qty| harga|total_penjualan|
+--------+-------------+-------------------+----------+---+------+---------------+
|TX7-0001|    Toko Solo|Kopi Susu Gula Aren|   Makanan|  7| 22000|         154000|
|TX7-0002|    Toko Solo|       Kemeja Batik|   Fashion|  5|250000|        1250000|
|TX7-0003|Toko Magelang|       Kemeja Batik|   Fashion|  6|250000|        1500000|
|TX7-0004|Toko Magelang|       Kemeja Batik|   Fashion|  6|250000|        1500000|
|TX7-0005|    Toko Solo|Blender Multifungsi|Elektronik|  3|450000|        1350000|
+--------+-------------+-------------------+----------+---+------+---------------+
only showing top 5 rows



## 🏆 5. TRANSFORM Analitik: Window Function (Top-3 Produk per Toko)

In [5]:
# 1. Agregasi total penjualan per kombinasi toko dan produk
ringkasan_produk_toko = df_gabungan.groupBy("nama_toko", "nama_produk").agg(
    F.sum("total_penjualan").alias("omset_produk")
)

# 2. Definisikan window partisi per toko urut omset menurun
window_toko = Window.partitionBy("nama_toko").orderBy(F.col("omset_produk").desc())

# 3. Beri nomor peringkat dan ambil Top-3
df_top3 = ringkasan_produk_toko \
    .withColumn("rank", F.row_number().over(window_toko)) \
    .filter(F.col("rank") <= 3) \
    .orderBy("nama_toko", "rank")

print("Top 3 Produk Terlaris di Masing-masing Toko Cabang:")
df_top3.show(12, truncate=False)

Top 3 Produk Terlaris di Masing-masing Toko Cabang:


+---------------+-------------------+------------+----+
|nama_toko      |nama_produk        |omset_produk|rank|
+---------------+-------------------+------------+----+
|Toko Magelang  |Smart TV 43 Inch   |131200000   |1   |
|Toko Magelang  |Blender Multifungsi|18900000    |2   |
|Toko Magelang  |Sepatu Sneakers    |17100000    |3   |
|Toko Semarang  |Smart TV 43 Inch   |105600000   |1   |
|Toko Semarang  |Blender Multifungsi|33300000    |2   |
|Toko Semarang  |Sepatu Sneakers    |22800000    |3   |
|Toko Solo      |Smart TV 43 Inch   |220800000   |1   |
|Toko Solo      |Kemeja Batik       |21500000    |2   |
|Toko Solo      |Blender Multifungsi|17100000    |3   |
|Toko Yogyakarta|Smart TV 43 Inch   |54400000    |1   |
|Toko Yogyakarta|Sepatu Sneakers    |17480000    |2   |
|Toko Yogyakarta|Blender Multifungsi|13050000    |3   |
+---------------+-------------------+------------+----+



## 🏛️ 6. TRANSFORM Analitik: Spark SQL (Kontribusi Omset per Manajer)

In [6]:
df_gabungan.createOrReplaceTempView("laporan_retail")

df_manager = spark.sql("""
    SELECT 
        kota,
        nama_toko,
        manager,
        COUNT(order_id) AS total_transaksi,
        SUM(total_penjualan) AS total_omset
    FROM laporan_retail
    GROUP BY kota, nama_toko, manager
    ORDER BY total_omset DESC
""")

print("Kontribusi Kinerja per Manajer Cabang:")
df_manager.show()

Kontribusi Kinerja per Manajer Cabang:


+----------+---------------+--------------+---------------+-----------+
|      kota|      nama_toko|       manager|total_transaksi|total_omset|
+----------+---------------+--------------+---------------+-----------+
|      Solo|      Toko Solo|  Bayu Nugroho|             81|  273424000|
|  Magelang|  Toko Magelang|Rani Handayani|             77|  182307000|
|  Semarang|  Toko Semarang|    Sari Indah|             80|  176739000|
|Yogyakarta|Toko Yogyakarta|   Joko Widodo|             62|   98464000|
+----------+---------------+--------------+---------------+-----------+



## 💾 7. Tahap LOAD — Menyimpan ke Parquet Terpartisi Kategori

In [7]:
output_hdfs = f"hdfs://localhost:9000/user/{username}/responsi/p7/processed/laporan_triwulan"
target_output = output_hdfs

try:
    df_gabungan.write.mode("overwrite") \
        .partitionBy("kategori") \
        .parquet(output_hdfs)
    print(f"✅ Laporan triwulan sukses disimpan ke HDFS: {output_hdfs}")
except Exception as e:
    print("⚠️ HDFS tidak aktif, menyimpan ke folder lokal 'demo_laporan_triwulan_parquet':")
    target_output = "demo_laporan_triwulan_parquet"
    df_gabungan.write.mode("overwrite") \
        .partitionBy("kategori") \
        .parquet(target_output)
    print(f"✅ Laporan triwulan disimpan ke lokal: {target_output}")

⚠️ HDFS tidak aktif, menyimpan ke folder lokal 'demo_laporan_triwulan_parquet':


✅ Laporan triwulan disimpan ke lokal: demo_laporan_triwulan_parquet


## 📊 8. Executive Summary & Verifikasi Akhir
Membaca kembali data Parquet untuk diserahkan ke jajaran direksi:

In [8]:
# Baca ulang dari target Parquet untuk memverifikasi integritas data
df_verifikasi = spark.read.parquet(target_output)

ringkasan_eksekutif = df_verifikasi.groupBy("nama_toko", "kota", "manager").agg(
    F.sum("total_penjualan").alias("total_omset_triwulan"),
    F.count("order_id").alias("banyak_transaksi"),
    F.round(F.avg("total_penjualan"), 0).alias("rata_belanja_per_transaksi")
).orderBy(F.col("total_omset_triwulan").desc())

print("=== RINGKASAN EKSEKUTIF DIREKSI RETAIL ===")
ringkasan_eksekutif.show()

# Bersihkan file lokal sementara
!rm -rf data_toko_p7.json data_produk_p7.csv data_transaksi_p7.csv demo_laporan_triwulan_parquet

# Tutup SparkSession
spark.stop()
print("🛑 Analisis End-to-End Selesai! SparkSession ditutup.")

=== RINGKASAN EKSEKUTIF DIREKSI RETAIL ===


+---------------+----------+--------------+--------------------+----------------+--------------------------+
|      nama_toko|      kota|       manager|total_omset_triwulan|banyak_transaksi|rata_belanja_per_transaksi|
+---------------+----------+--------------+--------------------+----------------+--------------------------+
|      Toko Solo|      Solo|  Bayu Nugroho|           273424000|              81|                 3375605.0|
|  Toko Magelang|  Magelang|Rani Handayani|           182307000|              77|                 2367623.0|
|  Toko Semarang|  Semarang|    Sari Indah|           176739000|              80|                 2209238.0|
|Toko Yogyakarta|Yogyakarta|   Joko Widodo|            98464000|              62|                 1588129.0|
+---------------+----------+--------------+--------------------+----------------+--------------------------+



🛑 Analisis End-to-End Selesai! SparkSession ditutup.
